# Typed Contracts in Python

A **contract** is a precise promise about data crossing a boundary — a function
call, a class, an API request, a file. A **typed contract** uses Python's type
system (and, when needed, runtime checks) to *state* and *enforce* that promise.

This notebook builds on the basics in
[`typing_type_hints_type_annotations.ipynb`](typing_type_hints_type_annotations.ipynb)
and is closely related to
[`dataclasses.ipynb`](dataclasses.ipynb),
[`pydantic.ipynb`](pydantic.ipynb) and
[`assertions.ipynb`](assertions.ipynb).

## 1. Two layers of a contract

| Layer | "Checked when?" | Tooling | Failure shows up as |
| ----- | --------------- | ------- | ------------------- |
| **Static** contract | *Before* running (analysis) | type hints + `mypy` / `pyright` | a type-checker error in your editor / CI |
| **Runtime** contract | *While* running | `isinstance`, `assert`, `pydantic`, `typeguard`/`beartype`, `icontract` | an exception (`TypeError`, `ValidationError`, ...) |

> **The crucial gotcha:** plain type hints are **not enforced at runtime**. The
> interpreter ignores them. They are a *specification* that static checkers
> verify — nothing stops wrong data at runtime unless you add a runtime check.

```python
def repeat(text: str, n: int) -> str:
    return text * n

# The hints say (str, int) -> str. A static checker (mypy/pyright) FLAGS the
# call below, but at runtime Python ignores the hints and runs it happily:
repeat(5, 3)        # -> 15  (int * int): violates the contract, yet NO error
print(repeat.__annotations__)
# {'text': <class 'str'>, 'n': <class 'int'>, 'return': <class 'str'>}
```

So a *typed contract* is the type hint **plus** the decision about where (if
anywhere) to enforce it at runtime.


## 2. Type hints as the contract specification

Hints document the contract for humans, editors, and static checkers.

```python
from collections.abc import Sequence

def mean(xs: Sequence[float]) -> float:
    # Pre: xs is non-empty.  Post: returns the arithmetic mean.
    return sum(xs) / len(xs)
```

Useful refinements that make the contract *narrower* (and let a checker catch
more bugs):

```python
from typing import Literal, Final, NewType, Annotated

Mode = Literal["r", "w", "a"]          # only these 3 strings are valid
def open_file(path: str, mode: Mode) -> None: ...
open_file("x.txt", "rw")               # mypy/pyright: error — "rw" not allowed

MAX_RETRIES: Final = 3                  # reassigning later is a type error

UserId = NewType("UserId", int)        # distinct from a plain int at type-check time
def load(uid: UserId) -> None: ...
load(42)                               # error: int is not UserId
load(UserId(42))                       # ok

# Annotated attaches machine-readable metadata to a type (used by pydantic, etc.)
Age = Annotated[int, "0..130"]
```


## 3. Structural contracts — `Protocol`

A **`Protocol`** says *"I don't care what class you are; I care what methods you
have."* This is **structural typing** (static duck typing) — the contract is a
shape, not an inheritance relationship.

```python
from typing import Protocol, runtime_checkable

class SupportsClose(Protocol):
    def close(self) -> None: ...

def shutdown(resource: SupportsClose) -> None:
    resource.close()

class Socket:                 # never inherits SupportsClose...
    def close(self) -> None:
        print("socket closed")

shutdown(Socket())            # ...but satisfies the contract structurally -> OK
```

* **Nominal** contract = `abc.ABC` + `isinstance`/subclassing (you must *declare*
  you implement it).
* **Structural** contract = `Protocol` (you implement it just by having the
  members). Great for decoupling and for typing third-party objects you can't
  subclass.

`@runtime_checkable` lets you use `isinstance` against a Protocol at runtime
(members only, not their signatures):

```python
@runtime_checkable
class SupportsClose(Protocol):
    def close(self) -> None: ...

isinstance(Socket(), SupportsClose)   # True
```


## 4. Shape contracts for dicts — `TypedDict`

`TypedDict` gives a static contract to a plain `dict` (common for JSON):

```python
from typing import TypedDict, NotRequired

class User(TypedDict):
    id: int
    name: str
    email: NotRequired[str]     # optional key (Python 3.11+)

u: User = {"id": 1, "name": "Ada"}            # ok
bad: User = {"id": "1", "name": "Ada"}        # checker error: id must be int
```

`TypedDict` is **static only** — at runtime it is just a `dict`, nothing is
validated. If the dict comes from the outside world, validate it (Section 6).


## 5. Why "static only" is not enough at boundaries

Static checks assume the *rest of your typed code* obeys the contract. They
cannot vouch for data that enters from **outside the type system**:

* JSON / HTTP request bodies, query params
* files, CSV, environment variables
* a database, a message queue, user input
* any `Any`-typed or untyped third-party value

At those **trust boundaries** you need a *runtime* contract that actually
inspects the data and rejects bad input. Inside well-typed code, static checks
usually suffice.

> **Rule of thumb:** *validate at the boundary, trust within.*


## 6. Runtime enforcement of type contracts

### 6a. Manual checks / `assert`
Cheapest, no dependency. `assert` is for *internal invariants you believe are
true* (and can be stripped with `python -O`); raise real exceptions for
*external* input.

```python
def set_age(age: int) -> None:
    if not isinstance(age, int) or not (0 <= age <= 130):
        raise ValueError(f"age out of contract: {age!r}")
```

### 6b. Enforce the annotations automatically — `typeguard` / `beartype`
These turn your *existing* type hints into runtime checks via a decorator.

```python
# pip install beartype
from beartype import beartype

@beartype
def repeat(text: str, n: int) -> str:
    return text * n

repeat("ab", 3)      # ok
repeat("ab", "3")    # BeartypeCallHintParamViolation at call time
```
`beartype` is O(1) (samples containers) and very fast; `typeguard` can do full
deep checks and offers an import hook. Use them on boundary functions or in
tests to catch contract violations the static checker can't prove.

### 6c. Validate + coerce structured data — `pydantic`
The standard for **data contracts** at API/IO boundaries: declare a model with
types/constraints; pydantic parses, validates, and coerces. See
[`pydantic.ipynb`](pydantic.ipynb).

```python
# pip install pydantic
from pydantic import BaseModel, Field, EmailStr

class User(BaseModel):
    id: int
    name: str = Field(min_length=1)
    age: int = Field(ge=0, le=130)

User(id="7", name="Ada", age=36)     # "7" -> 7, validated -> ok
User(id=1, name="", age=200)         # ValidationError: name too short, age too big
```
`pydantic.validate_call` does the same for a function's arguments:

```python
from pydantic import validate_call

@validate_call
def area(w: float, h: float) -> float:
    return w * h
area("3", 4)     # coerced -> 12.0
```

### 6d. Dataclasses + `__post_init__`
A lightweight, dependency-free runtime check for your own value objects
(see [`dataclasses.ipynb`](dataclasses.ipynb)):

```python
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float
    def __post_init__(self) -> None:
        if not all(isinstance(v, (int, float)) for v in (self.x, self.y)):
            raise TypeError("Point coordinates must be numbers")
```


## 7. Design by Contract (DbC): preconditions, postconditions, invariants

"Typed contracts" generalize to **Design by Contract** (Bertrand Meyer, Eiffel):
every function/class declares obligations and guarantees.

* **Precondition** (`require`) — what the *caller* must guarantee on entry.
* **Postcondition** (`ensure`) — what the *function* guarantees on exit.
* **Invariant** — what stays true for an object between method calls.

### 7a. Plain `assert` version (no dependency)
```python
def sqrt(x: float) -> float:
    assert x >= 0, "precondition: x must be non-negative"     # require
    r = x ** 0.5
    assert abs(r * r - x) < 1e-9, "postcondition failed"      # ensure
    return r
```
(Asserts vanish under `python -O`; use them for *internal* contracts, not for
validating untrusted input.)

### 7b. The `icontract` library — explicit, introspectable contracts
```python
# pip install icontract
import icontract

@icontract.require(lambda x: x >= 0, "x must be non-negative")
@icontract.ensure(lambda result, x: abs(result * result - x) < 1e-9)
def sqrt(x: float) -> float:
    return x ** 0.5

@icontract.invariant(lambda self: self.balance >= 0)
class Account:
    def __init__(self) -> None:
        self.balance = 0
    def withdraw(self, amount: int) -> None:
        self.balance -= amount      # violating the invariant raises ViolationError
```
`icontract` keeps contracts next to the code, includes them in docs, and can be
toggled off in production.


## 8. When to use which

| Need | Use |
| ---- | --- |
| Document intent + catch bugs early, zero runtime cost | **Type hints** + `mypy`/`pyright` |
| "Has the right shape" without inheritance | **`Protocol`** (add `@runtime_checkable` for `isinstance`) |
| Static contract for a JSON-ish dict | **`TypedDict`** |
| Restrict to specific values / constants / distinct ids | **`Literal`**, **`Final`**, **`NewType`**, **`Annotated`** |
| Validate & coerce **external** data (API, config, IO) | **`pydantic`** (`BaseModel`, `validate_call`) |
| Enforce existing hints at runtime (boundaries, tests) | **`beartype`** / **`typeguard`** |
| Lightweight runtime check for your own value objects | **`dataclass` + `__post_init__`** |
| Pre/post-conditions & class invariants on critical logic | **`assert`** (internal) or **`icontract`** (formal DbC) |

**Layering that works in practice**
1. Type-hint everything; run `mypy`/`pyright` in CI (static contract).
2. At every trust boundary, validate with `pydantic` (or beartype/typeguard).
3. For critical algorithms/invariants, add DbC (`assert` / `icontract`).
4. Trust your types *inside* the validated core — don't re-check everywhere.

### References
* PEP 484 (type hints), PEP 544 (`Protocol`), PEP 589 (`TypedDict`),
  PEP 586 (`Literal`), PEP 593 (`Annotated`), PEP 681 (dataclass transforms)
* pydantic: https://docs.pydantic.dev/
* beartype: https://github.com/beartype/beartype ·
  typeguard: https://typeguard.readthedocs.io/
* icontract: https://icontract.readthedocs.io/ ·
  Design by Contract (Meyer): https://en.wikipedia.org/wiki/Design_by_contract
* Static checkers: see
  [`type_checkers_mypy_pyright_ruff.ipynb`](type_checkers_mypy_pyright_ruff.ipynb)
